In [1]:
!pip install -q langchain langchain-groq gspread

In [2]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

print("Cafe Napoli Groq connected successfully!")

Cafe Napoli Groq connected successfully!


In [3]:
MENU = {
    "espresso": 100,
    "cappuccino": 150,
    "latte": 160,
    "cold coffee": 140,
    "mocha": 180,

    "chocolate cake": 180,
    "cheesecake": 200,
    "brownie": 120,
    "tiramisu": 220,

    "mango smoothie": 130,
    "strawberry smoothie": 150,
    "banana smoothie": 120,

    "margherita pizza": 250,
    "farmhouse pizza": 300,
    "mexican pizza": 320,
    "veg quesadilla": 200,
    "nachos": 150
}

print("Cafe Napoli menu loaded successfully!")
print("Total menu items:", len(MENU))

Cafe Napoli menu loaded successfully!
Total menu items: 17


In [4]:
from langchain.tools import tool

@tool
def get_cafe_price(item: str) -> str:
    """Get the price of a Cafe Napoli menu item.

    Args:
        item: Name of the food or drink item.
    """
    item = item.lower().strip()

    if item not in MENU:
        return f"Sorry, {item} is not available on our menu."

    return f"The price of {item} is Rs. {MENU[item]}."

print("get_cafe_price tool created successfully!")

get_cafe_price tool created successfully!


In [5]:
@tool
def check_cafe_item(item: str) -> str:
    """Check whether a food or drink item is available at Cafe Napoli.

    Args:
        item: Name of the menu item.
    """
    item = item.lower().strip()

    if item in MENU:
        return f"Yes, {item} is available at Cafe Napoli."

    return f"Sorry, {item} is not available at Cafe Napoli."

print("check_cafe_item tool created successfully!")

check_cafe_item tool created successfully!


In [7]:
from pydantic import BaseModel, Field
from typing import Optional

class CafeOrder(BaseModel):
    name: Optional[str] = Field(
        default=None,
        description="Customer name"
    )

    item: Optional[str] = Field(
        default=None,
        description="Food or drink item"
    )

    quantity: Optional[int] = Field(
        default=None,
        description="Quantity ordered"
    )

    pickup_date: Optional[str] = Field(
        default=None,
        description="Pickup date"
    )

    eggless: Optional[bool] = Field(
        default=None,
        description="Whether eggless is requested"
    )

print("Cafe Napoli Order model created successfully! ")

Cafe Napoli Order model created successfully! 


In [8]:
structured_llm = llm.with_structured_output(CafeOrder)

print("Structured output connected successfully!")

Structured output connected successfully!


In [9]:
messages = [
    "Hey, I'm Ravi. Need 2 cappuccinos tomorrow.",
    "plz book one choclate cake for priya on sunday",
    "I want 3 pizzas, name is Rahul, pickup next Friday",
    "Can I get a eggless brownie? I'm Sneha",
    "2 cold coffees please for tonight"
]

for msg in messages:
    print("\nCustomer:", msg)

    result = structured_llm.invoke(msg)

    print("Structured Output:")
    print(result.model_dump())


Customer: Hey, I'm Ravi. Need 2 cappuccinos tomorrow.
Structured Output:
{'name': 'Ravi', 'item': 'cappuccino', 'quantity': 2, 'pickup_date': 'tomorrow', 'eggless': False}

Customer: plz book one choclate cake for priya on sunday
Structured Output:
{'name': 'Priya', 'item': 'chocolate cake', 'quantity': 1, 'pickup_date': 'Sunday', 'eggless': None}

Customer: I want 3 pizzas, name is Rahul, pickup next Friday
Structured Output:
{'name': 'Rahul', 'item': 'pizza', 'quantity': 3, 'pickup_date': '2026-11-28', 'eggless': None}

Customer: Can I get a eggless brownie? I'm Sneha
Structured Output:
{'name': 'Sneha', 'item': 'brownie', 'quantity': 1, 'pickup_date': None, 'eggless': True}

Customer: 2 cold coffees please for tonight
Structured Output:
{'name': None, 'item': 'cold coffee', 'quantity': 2, 'pickup_date': 'tonight', 'eggless': None}


In [10]:
from google.colab import auth
import gspread
from google.auth import default

auth.authenticate_user()

creds, _ = default()
gc = gspread.authorize(creds)

sh = gc.create("Cafe Napoli Orders")
ws = sh.sheet1

ws.append_row([
    "name",
    "item",
    "quantity",
    "pickup_date",
    "eggless",
    "total"
])

print("Cafe Napoli Google Sheet created!")
print("Sheet URL:", sh.url)

Cafe Napoli Google Sheet created!
Sheet URL: https://docs.google.com/spreadsheets/d/11kR1CbIjmOA6gR_-hIFnnUVqRvPtHTbaJNxhHnKDi7o


In [11]:
@tool
def log_cafe_order(
    name: str,
    item: str,
    quantity: int,
    pickup_date: str,
    eggless: str = "Not specified"
) -> str:
    """Save a complete Cafe Napoli customer order to Google Sheets.

    Args:
        name: Customer name.
        item: Exact food or drink item from the menu.
        quantity: Number of items ordered.
        pickup_date: Confirmed pickup date.
        eggless: Yes, No, or Not specified.
    """
    item = item.lower().strip()

    if not name.strip():
        return "Please provide the customer name."

    if item not in MENU:
        return f"Sorry, {item} is not an exact menu item. Please select an item from our menu."

    if quantity <= 0:
        return "Quantity must be greater than zero."

    if not pickup_date.strip():
        return "Please provide a pickup date."

    total = MENU[item] * quantity

    ws.append_row([
        name.strip(),
        item,
        quantity,
        pickup_date,
        eggless,
        total
    ])

    return f"Order saved successfully! Total amount: Rs. {total}."

print("Cafe Napoli order logging tool created successfully! ")

Cafe Napoli order logging tool created successfully! 


In [12]:
from langchain.agents import create_agent

SYSTEM = """
You are the friendly AI assistant of Cafe Napoli.

Your responsibilities:
1. Help customers with menu items and prices.
2. Check whether an item is available using the tools.
3. Take customer orders and save them using log_cafe_order.

Important rules:
- Only use items present in the MENU.
- Never invent menu items or prices.
- If a customer says only "pizza", ask which pizza they want.
- Ask for missing customer name, quantity, or pickup date.
- Quantity must be greater than zero.
- Confirm unclear pickup dates before saving an order.
- Never save an incomplete or unconfirmed order.
- Do not claim an order is saved unless the logging tool succeeds.
- Be polite, friendly, and concise.
"""

tools = [
    get_cafe_price,
    check_cafe_item,
    log_cafe_order
]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM
)

print("Cafe Napoli AI Agent created successfully!")

Cafe Napoli AI Agent created successfully!


In [13]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Hi! I'm Rahul. What is the price of a cappuccino?"
        }
    ]
})

for message in response["messages"]:
    if message.type == "ai" and message.content:
        print(message.content)

Hi Rahul! The cappuccino is Rs. 150. Would you like to place an order? If so, just let me know the item, quantity, and pickup date.


In [14]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "I'm Priya. I want 2 chocolate cakes for tomorrow. Please book my order."
        }
    ]
})

for message in response["messages"]:
    if message.type == "ai" and message.content:
        print(message.content)

Sure thing, Priya! Just to confirm, you’d like to pick up your 2 chocolate cakes tomorrow (i.e., [date]). Is that correct?


In [15]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Yes, that's correct. Please confirm my order for 28 November 2026."
        }
    ]
})

for message in response["messages"]:
    if message.type == "ai" and message.content:
        print(message.content)

Sure thing! I just need a few details to make sure everything’s set:

1. **Your name** – so we can address you properly.  
2. **The item(s) you’d like to order** – let me know which menu item(s) you’re picking.  
3. **Quantity** – how many of each item?  

You mentioned a pickup date of **28 November 2026** – that’s noted. Once I have the rest of the info, I’ll confirm the order for you.


In [16]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": """
            My name is Priya.
            I want 2 chocolate cakes.
            My pickup date is 28 November 2026.
            Please save my order.
            """
        }
    ]
})

for message in response["messages"]:
    if message.type == "ai" and message.content:
        print(message.content)

✅ Your order has been saved!  
- **Name:** Priya  
- **Item:** 2 × Chocolate Cake  
- **Pickup Date:** 28 November 2026  

Total amount: **Rs. 360**.  
Thank you for choosing Cafe Napoli! Let us know if there’s anything else you’d like.


In [17]:
records = ws.get_all_records()

for record in records:
    print(record)

{'name': 'Priya', 'item': 'chocolate cake', 'quantity': 2, 'pickup_date': '28 November 2026', 'eggless': 'Not specified', 'total': 360}


In [18]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Hi, I want 2 chicken burgers. Please tell me the price."
        }
    ]
})

for message in response["messages"]:
    if message.type == "ai" and message.content:
        print(message.content)

I’m sorry, but “chicken burger” isn’t on our menu. Could you let me know which item you’d like instead? Once we have the correct item, I’ll check the price for you. Also, could you please share your name and the pickup date you’re planning? That way I can place the order for you.


In [19]:
tool1 = get_cafe_price.model_copy(update={"name": "tool1"})

print("Original tool:", get_cafe_price.name)
print("Renamed tool:", tool1.name)


Original tool: get_cafe_price
Renamed tool: tool1


In [20]:
agent_renamed = create_agent(
    model=llm,
    tools=[
        tool1,
        check_cafe_item,
        log_cafe_order
    ],
    system_prompt=SYSTEM
)

print("Temporary agent created with renamed tool!")

Temporary agent created with renamed tool!


In [21]:
response = agent_renamed.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the price of a latte?"
        }
    ]
})

for message in response["messages"]:
    if message.type == "ai" and message.content:
        print(message.content)

The latte is Rs. 160. Let me know if you’d like to order or need anything else!


In [22]:
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

What is the price of a latte?
================================== Ai Message ==================================
Tool Calls:
  tool1 (fc_b59cbda9-72cb-4f05-ad90-f18770292994)
 Call ID: fc_b59cbda9-72cb-4f05-ad90-f18770292994
  Args:
    item: latte
================================= Tool Message =================================
Name: tool1

The price of latte is Rs. 160.
================================== Ai Message ==================================

The latte is Rs. 160. Let me know if you’d like to order or need anything else!


In [23]:
agent = create_agent(
    model=llm,
    tools=[
        get_cafe_price,
        check_cafe_item,
        log_cafe_order
    ],
    system_prompt=SYSTEM
)

print("Original Cafe Napoli agent restored successfully!")

Original Cafe Napoli agent restored successfully!


In [24]:
facts = """
CAFE NAPOLI - BUSINESS FACTS

1. Business name: Cafe Napoli.
2. Cafe Napoli is a cafe business.
3. The cafe serves coffee and other beverages.
4. The cafe offers desserts and cakes.
5. The cafe offers pizza and Mexican-style food.
6. Cappuccino is one of the menu items.
7. Chocolate cake is one of the menu items.
8. Cold coffee is one of the menu items.
9. Tiramisu is one of the dessert options.
10. Mexican pizza is one of the food options.
11. The cafe has staff management and retention challenges.
12. Staff work consistency needs improvement.
13. Equipment and infrastructure issues are sometimes handled reactively.
14. The cafe needs better equipment maintenance planning.
15. The cafe needs improvement in marketing and online presence.
16. Standard Operating Procedures (SOPs) are not yet properly available for the hot kitchen.
17. The cafe needs standardized kitchen processes to improve consistency.
18. AI automation could help with customer orders, staff operations, maintenance tracking, and marketing.

Note: Menu items and prices used in the workshop prototype are sample data.
They must be verified with Cafe Napoli before real customer use.
"""

with open("cafe_napoli_facts.txt", "w", encoding="utf-8") as f:
    f.write(facts)

print("cafe_napoli_facts.txt created successfully! ")
print(facts)

cafe_napoli_facts.txt created successfully! 

CAFE NAPOLI - BUSINESS FACTS

1. Business name: Cafe Napoli.
2. Cafe Napoli is a cafe business.
3. The cafe serves coffee and other beverages.
4. The cafe offers desserts and cakes.
5. The cafe offers pizza and Mexican-style food.
6. Cappuccino is one of the menu items.
7. Chocolate cake is one of the menu items.
8. Cold coffee is one of the menu items.
9. Tiramisu is one of the dessert options.
10. Mexican pizza is one of the food options.
11. The cafe has staff management and retention challenges.
12. Staff work consistency needs improvement.
13. Equipment and infrastructure issues are sometimes handled reactively.
14. The cafe needs better equipment maintenance planning.
15. The cafe needs improvement in marketing and online presence.
16. Standard Operating Procedures (SOPs) are not yet properly available for the hot kitchen.
17. The cafe needs standardized kitchen processes to improve consistency.
18. AI automation could help with custo

In [25]:
from google.colab import files
files.download("cafe_napoli_facts.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>